In [1]:
import pandas as pd
import numpy as np
class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _to_html(self, obj):
        if hasattr(obj, '_repr_html_'):
            html = obj._repr_html_()
            if html is not None:
                return html
        if hasattr(obj, 'to_html'):
            return obj.to_html()
        return f"<pre>{repr(obj)}</pre>"
        
    def _repr_html_(self):
        rendered = []
        for a in self.args:
            try:
                obj = eval(a)
                rendered.append(self.template.format(a, self._to_html(obj)))
            except Exception as e:
                rendered.append(self.template.format(a, f"<pre>Error: {e}</pre>"))
        return '\n'.join(rendered)
    
    def __repr__(self):
        lines = []
        for a in self.args:
            try:
                lines.append(a + '\n' + repr(eval(a)))
            except Exception as e:
                lines.append(f"{a}\nError: {e}")
        return '\n\n'.join(lines)

panel: 3d data.
panel4d: 4d data.
hierarchical indexing is a powerful feature in pandas that allows you to work with multi-level or multi-dimensional data in a more intuitive way. It enables you to have multiple index levels on rows and/or columns, which can be useful for representing complex datasets.

# A multiple indexed series

In [2]:
index = [('Toyota', 2010), ('Toyota', 2011), ('Toyota', 2012), ('Honda', 2010), ('Honda', 2011), ('Honda', 2012)]
models = ['Camry', 'Corolla', 'Prius', 'Civic', 'Accord', 'Fit']
series = pd.Series(models, index=index)
print(series, "\n")
print(series[('Toyota', 2012):('Honda', 2011)])
try:
    print(series['Toyota'])
except KeyError as e:
    print(f"KeyError: {e}")
try:
    print(series[2011])
except KeyError as e:
    print(f"KeyError: {e}")

(Toyota, 2010)      Camry
(Toyota, 2011)    Corolla
(Toyota, 2012)      Prius
(Honda, 2010)       Civic
(Honda, 2011)      Accord
(Honda, 2012)         Fit
dtype: str 

(Toyota, 2012)     Prius
(Honda, 2010)      Civic
(Honda, 2011)     Accord
dtype: str
KeyError: 'Toyota'
KeyError: 2011


## pandas multiindex

In [3]:
index = pd.MultiIndex.from_tuples(index)
series = series.reindex(index)
print(series, "\n")
print(series['Toyota', :], "\n")
print(series[:, 2011])

Toyota  2010      Camry
        2011    Corolla
        2012      Prius
Honda   2010      Civic
        2011     Accord
        2012        Fit
dtype: str 

2010      Camry
2011    Corolla
2012      Prius
dtype: str 

Toyota    Corolla
Honda      Accord
dtype: str


## multiindex as extra dimension

In [4]:
display('series', "series.unstack()")

,2010,2011,2012
Honda,Civic,Accord,Fit
Toyota,Camry,Corolla,Prius


In [5]:
costs = [13123, 242243, 124234, 98763543, 2345, 2135]
df = pd.DataFrame({'model': series, 'cost': costs})
print(df)

               model      cost
Toyota 2010    Camry     13123
       2011  Corolla    242243
       2012    Prius    124234
Honda  2010    Civic  98763543
       2011   Accord      2345
       2012      Fit      2135


### creation of multiindex

In [6]:
display('pd.MultiIndex.from_arrays([["Toyota", "Toyota", "Honda", "Honda"], [2010, 2011, 2010, 2011]])',
        'pd.MultiIndex.from_product([["Toyota", "Honda"], [2010, 2011]])',
        'pd.MultiIndex.from_tuples([("Toyota", 2010), ("Toyota", 2011), ("Honda", 2010), ("Honda", 2011)])')

pd.MultiIndex.from_arrays([["Toyota", "Toyota", "Honda", "Honda"], [2010, 2011, 2010, 2011]])
MultiIndex([('Toyota', 2010),
            ('Toyota', 2011),
            ( 'Honda', 2010),
            ( 'Honda', 2011)],
           )

pd.MultiIndex.from_product([["Toyota", "Honda"], [2010, 2011]])
MultiIndex([('Toyota', 2010),
            ('Toyota', 2011),
            ( 'Honda', 2010),
            ( 'Honda', 2011)],
           )

pd.MultiIndex.from_tuples([("Toyota", 2010), ("Toyota", 2011), ("Honda", 2010), ("Honda", 2011)])
MultiIndex([('Toyota', 2010),
            ('Toyota', 2011),
            ( 'Honda', 2010),
            ( 'Honda', 2011)],
           )

## names of the index levels

In [7]:
print(df, '\n')
df.index.names = ['make', 'year']
print(df)

               model      cost
Toyota 2010    Camry     13123
       2011  Corolla    242243
       2012    Prius    124234
Honda  2010    Civic  98763543
       2011   Accord      2345
       2012      Fit      2135 

               model      cost
make   year                   
Toyota 2010    Camry     13123
       2011  Corolla    242243
       2012    Prius    124234
Honda  2010    Civic  98763543
       2011   Accord      2345
       2012      Fit      2135


In [8]:
models = df['model']
trims = ['LE', 'SE', 'XLE', 'LX', 'EX', 'Sport']
cost = df['cost']
columns = ['type', 'cost']
columns_final = pd.MultiIndex.from_product([['type', 'cost'], ['full', 'part']], names=columns)
mx_cost = [523, 234, 1234, 2345, 3456, 4567]

new_df = pd.DataFrame(
{
('type', 'full'): models,
('type', 'part'): trims,
('cost', 'full'): cost,
('cost', 'part'): mx_cost
},
index=index
).reindex(columns=columns_final)

display('new_df')

new_df
type            type             cost      
cost            full   part      full  part
Toyota 2010    Camry     LE     13123   523
       2011  Corolla     SE    242243   234
       2012    Prius    XLE    124234  1234
Honda  2010    Civic     LX  98763543  2345
       2011   Accord     EX      2345  3456
       2012      Fit  Sport      2135  4567

# indexing and slicing with multiindex

In [9]:
display('series_sorted', 'series_sorted["Toyota"]', 'series_sorted["Toyota", :]', 'series_sorted.loc[("Honda", 2011):("Toyota", 2011)]')

series_sorted
Error: name 'series_sorted' is not defined

series_sorted["Toyota"]
Error: name 'series_sorted' is not defined

series_sorted["Toyota", :]
Error: name 'series_sorted' is not defined

series_sorted.loc[("Honda", 2011):("Toyota", 2011)]
Error: name 'series_sorted' is not defined

## multiply indexed dataframe

In [16]:

display('new_df', 'new_df["type", "full"]', 'new_df.iloc[1:, :-1]')

new_df
type            type             cost      
cost            full   part      full  part
Toyota 2010    Camry     LE     13123   523
       2011  Corolla     SE    242243   234
       2012    Prius    XLE    124234  1234
Honda  2010    Civic     LX  98763543  2345
       2011   Accord     EX      2345  3456
       2012      Fit  Sport      2135  4567

new_df["type", "full"]
Toyota  2010      Camry
        2011    Corolla
        2012      Prius
Honda   2010      Civic
        2011     Accord
        2012        Fit
Name: (type, full), dtype: str

new_df.iloc[1:, :-1]
type            type             cost
cost            full   part      full
Toyota 2011  Corolla     SE    242243
       2012    Prius    XLE    124234
Honda  2010    Civic     LX  98763543
       2011   Accord     EX      2345
       2012      Fit  Sport      2135

# rearranging multi-indices

In [11]:
print(new_df, "\n")
temp = new_df.sort_index()
print(temp, "\n")
temp = new_df.sort_index(level=1)
print(temp)

type            type             cost      
cost            full   part      full  part
Toyota 2010    Camry     LE     13123   523
       2011  Corolla     SE    242243   234
       2012    Prius    XLE    124234  1234
Honda  2010    Civic     LX  98763543  2345
       2011   Accord     EX      2345  3456
       2012      Fit  Sport      2135  4567 

type            type             cost      
cost            full   part      full  part
Honda  2010    Civic     LX  98763543  2345
       2011   Accord     EX      2345  3456
       2012      Fit  Sport      2135  4567
Toyota 2010    Camry     LE     13123   523
       2011  Corolla     SE    242243   234
       2012    Prius    XLE    124234  1234 

type            type             cost      
cost            full   part      full  part
Honda  2010    Civic     LX  98763543  2345
Toyota 2010    Camry     LE     13123   523
Honda  2011   Accord     EX      2345  3456
Toyota 2011  Corolla     SE    242243   234
Honda  2012      Fit  Sport 

## index resetting

In [12]:
print(df, "\n")
print(df.reset_index())

               model      cost
make   year                   
Toyota 2010    Camry     13123
       2011  Corolla    242243
       2012    Prius    124234
Honda  2010    Civic  98763543
       2011   Accord      2345
       2012      Fit      2135 

     make  year    model      cost
0  Toyota  2010    Camry     13123
1  Toyota  2011  Corolla    242243
2  Toyota  2012    Prius    124234
3   Honda  2010    Civic  98763543
4   Honda  2011   Accord      2345
5   Honda  2012      Fit      2135


# data aggregations on multiindices

In [18]:
display('df', 'df.groupby(level="year").mean(numeric_only=True)')

df
               model      cost
make   year                   
Toyota 2010    Camry     13123
       2011  Corolla    242243
       2012    Prius    124234
Honda  2010    Civic  98763543
       2011   Accord      2345
       2012      Fit      2135

df.groupby(level="year").mean(numeric_only=True)
            cost
year            
2010  49388333.0
2011    122294.0
2012     63184.5